# Étape 7 — Couche Risque (VaR, Expected Shortfall, vol GARCH, régime de risque)
## MASI Hybrid Forecasting System

Ajout d'une **protection ex-ante** au-dessus du signal CNN-LSTM `base12` (modèle de production validé étapes 5/6). On NE MODIFIE PAS le prédicteur — on ajoute trois variantes de filtre :

1. **Filtre VaR paramétrique 5%** (μ + σ_GARCH · Φ⁻¹(0.05)) — bloque si `y_pred < VaR`
2. **Filtre ES paramétrique 5%** (Expected Shortfall) — plus strict que VaR
3. **Filtre régime de risque** (low/normal/high via quantiles σ_GARCH figés TRAIN+VAL) — stand-aside si vol haute

Validation : Kupiec POF (taux de breach) + Christoffersen (indépendance des breaches).

## 1. Exécution du script (génère etape6_final_predictions.csv + métriques risque + plots)

In [ ]:
import sys, os, json
sys.path.insert(0, os.path.abspath('.'))

import pandas as pd
from IPython.display import Image, display

import importlib.util as _ilu, sys as _sys; _spec = _ilu.spec_from_file_location('e7', '../scripts/07_risk_layer.py'); e7 = _ilu.module_from_spec(_spec); _sys.modules['e7'] = e7; _spec.loader.exec_module(e7)
out, validation = e7.main()

## 2. Fichier canonique étape 6 — `etape6_final_predictions.csv`

Une ligne par jour TEST, format API/dashboard :
`date, actual_return, predicted_return, signal, regime, regime_name, strategy_return`.

In [ ]:
final = pd.read_csv('../outputs/etape6/etape6_final_predictions.csv', parse_dates=['date'])
print(f'Shape : {final.shape}')
print(f'Période : {final.date.min().date()} → {final.date.max().date()}')
print(f'Somme strategy_return : {final.strategy_return.sum():+.4f} (≈ ln(equity finale CNN-LSTM))')
final.head()

## 3. Validation backtest VaR — Kupiec POF + Christoffersen

In [ ]:
v = validation
rows = []
for kind in ['historical', 'parametric']:
    k = v['kupiec'][kind]
    c = v['christoffersen'][kind]
    rows.append({
        'VaR type': kind,
        'breaches': f"{k['x']}/{k['T']}",
        'taux obs (%)': round(k['p_obs']*100, 2),
        'Kupiec p': round(k['pvalue'], 3) if k['pvalue'] else None,
        'Kupiec verdict': k['verdict'],
        'Christoffersen p': round(c['pvalue'], 3) if c['pvalue'] else None,
        'Christoffersen verdict': c['verdict'],
    })
pd.DataFrame(rows)

## 4. Comparaison des stratégies (Sharpe / MDD / equity finale)

In [ ]:
rows = []
for k, d in validation['strategy_comparison'].items():
    rows.append({
        'stratégie': k,
        'Sharpe ann.': round(d['sharpe'], 3),
        'MDD': f"{d['mdd']*100:+.2f}%",
        'eq finale': round(d['final_equity'], 3),
        'n trades': d['n_trades'],
    })
pd.DataFrame(rows).set_index('stratégie')

## 5. Répartition du régime de risque sur TEST

In [ ]:
rc = validation['risk_regime_counts_test']
total = sum(rc.values())
print(f"Seuils figés TRAIN+VAL : q33={validation['risk_regime_thresholds']['q33']:.5f} | q67={validation['risk_regime_thresholds']['q67']:.5f}")
pd.DataFrame([{'régime': k, 'n jours': v, '%': round(v/total*100, 1)} for k, v in rc.items()]).sort_values('n jours', ascending=False)

## 6. Résumé VaR/ES (moyennes TEST)

In [ ]:
ve = validation['var_es_summary']
pd.DataFrame([{'métrique': k, 'valeur': round(v, 5)} for k, v in ve.items()])

## 7. Plots

In [ ]:
Image(os.path.join('../reports/figures/etape7', 'etape7_vol_cone.png'))

In [ ]:
Image(os.path.join('../reports/figures/etape7', 'etape7_var_breaches.png'))

In [ ]:
Image(os.path.join('../reports/figures/etape7', 'etape7_return_distribution.png'))

In [ ]:
Image(os.path.join('../reports/figures/etape7', 'etape7_mdd_comparison.png'))

## 8. Conclusion (à approfondir dans `outputs/etape7/report.md`)

- Les **VaR 5%** (hist & param) passent Kupiec (taux de breach conforme à 5%) mais **échouent Christoffersen** : les breaches forment des clusters temporels — limite connue d'un VaR statique pendant les épisodes de stress.
- Les **filtres VaR/ES** sur `y_pred` ne déclenchent JAMAIS (les prédictions CNN-LSTM sont trop conservatrices, |y_pred| << |VaR|). Honnêtement : ces filtres n'ajoutent rien à ce signal.
- Le **filtre régime de risque** (stand-aside si vol haute) est le seul qui apporte une vraie valeur : **MDD divisé par 2** (−16 % → −7 %) et **Sharpe amélioré** (+1.06 → +1.20), au prix d'une equity finale plus basse (1.42 vs 1.74).

→ Conclusion : le **filtre régime de risque** est la couche utile. Sera testé en combinaison avec le régime HMM directionnel à l'étape 8.